In [1]:
import pandas as pd
import igraph
import pickle as pk
import os

In [38]:
print("Available network graph files:")
for g in set( [g.split('_')[0] for g in os.listdir("graphs/")] ):
    print("-", g)

Available network graph files:
- LitLangMed
- Humanities


In [9]:
dataset = 'LitLangMed'

# Create cocitation graph

In [2]:
def describe(g):
    print( g.vcount() )
    print( g.ecount() )
    print( g.vertex_attributes() )
    print( g.edge_attributes() )

In [10]:
## Load graph
g = igraph.Graph.Read_Pickle( 'graphs/' + dataset + '_graph.pk' )
describe(g)

5298
7017
['name', 'title', 'school', 'type', 'authors']
[]


In [11]:
cocitation_matrix = g.cocitation()

In [12]:
gc = igraph.Graph.Weighted_Adjacency(
    cocitation_matrix,
    mode='undirected',
    attr='weight',
    loops=False
)

for attr in g.vertex_attributes():
    gc.vs[attr] = g.vs[attr]

In [13]:
describe(gc)

5298
892254
['name', 'title', 'school', 'type', 'authors']
['weight']


In [14]:
## Write to pickle
gc.write_pickle( "graphs/" + dataset + "_cocitation_graph.pk")

# Visualize

In [15]:
from fa2_modified import ForceAtlas2
import random

In [23]:
gc = igraph.Graph.Read_Pickle( 'graphs/' + dataset + '_cocitation_graph.pk' )

### Filter and layout

In [30]:
filt_thresh = 2

In [31]:
# Delete edges with cocitation weight less than threshold
gc.es.select(weight_lt = filt_thresh).delete()

# Delete vertices with zero degree
gc.vs.select(_degree_lt=1).delete()

print( gc.vcount() )
print( gc.ecount() )

5278
61385


In [32]:
random.seed(1327)

forceatlas2 = ForceAtlas2(
                        # Behavior alternatives
                        outboundAttractionDistribution=False,  # Dissuade hubs
                        linLogMode=False,  # NOT IMPLEMENTED
                        adjustSizes=False,  # Prevent overlap (NOT IMPLEMENTED)
                        edgeWeightInfluence=1.5,

                        # Performance
                        jitterTolerance=1.0,  # Tolerance
                        barnesHutOptimize=True,
                        barnesHutTheta=1.2,
                        multiThreaded=False,  # NOT IMPLEMENTED

                        # Tuning
                        scalingRatio=1.0,
                        strongGravityMode=True,
                        gravity=2,

                        # Log
                        verbose=True)

l = forceatlas2.forceatlas2_igraph_layout( gc, pos=None, iterations=1000, weight_attr='weight' )

100%|██████████████████████████████████████████████████████████| 1000/1000 [01:00<00:00, 16.41it/s]

BarnesHut Approximation  took  24.88  seconds
Repulsion forces  took  32.87  seconds
Gravitational forces  took  0.02  seconds
Attraction forces  took  1.82  seconds
AdjustSpeedAndApplyForces step  took  0.61  seconds


In [33]:
# Write layout
with open("layouts/" + dataset + "_" + str(filt_thresh) +  ".pkl", "wb") as f:
    pk.dump(l, f)

### Plot

In [34]:
with open("layouts/" + dataset + "_" + str(filt_thresh) + ".pkl", "rb") as f:
    l = pk.load(f)

In [35]:
igraph.plot(
    gc,
    'plots/' + dataset + "_" + str(filt_thresh) + '_cocitations.png', 
    layout=l,
    vertex_size = 1,
    #vertex_size = [(s/200)+2 for s in gc.vs['strength']],
    edge_width=0.1
)